# Day 6: Real-Time PySpark Structured Streaming + MLlib + ELK
### End-to-End IoT Anomaly Detection Pipeline

```
[Kafka: station-telemetry] 
       ↓ (Structured Streaming Micro-batches)
[Spark MLlib: Feature Assembly & Inference in RAM]
       ↓ (Critical Fault Filtering)
[Kafka: station-alerts] → [Logstash Pipeline] → [Elasticsearch: station-alerts] → [Kibana Dashboard]
```

In this notebook:
1. Connect to live Kafka stream `station-telemetry`.
2. Load our pre-trained MLlib anomaly classification model.
3. Apply model scoring across incoming micro-batches.
4. Stream alerts to Kafka `station-alerts` & bulk index directly into Elasticsearch.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, struct, to_json, lit
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.ml import PipelineModel
import os

kafka_server = "kafka:29092"
input_topic = "station-telemetry"
output_topic = "station-alerts"
if os.path.exists("/app/models/ev_anomaly_model"):
    model_path = "/app/models/ev_anomaly_model"
elif os.path.exists("/workspace/models/ev_anomaly_model"):
    model_path = "/workspace/models/ev_anomaly_model"
else:
    model_path = "./models/ev_anomaly_model"

# Start Spark with pre-configured Kafka connectors
spark = SparkSession.builder \
    .appName("EV-Streaming-Anomaly-Detection") \
    .master("local[2]") \
    .config("spark.driver.memory", "512m") \
    .config("spark.sql.streaming.checkpointLocation", "/tmp/checkpoints/streaming_app") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark Session initialized! Ready for Structured Streaming.")

try:
    from elasticsearch import Elasticsearch
    es = Elasticsearch("http://elasticsearch:9200")
    template_body = {
        "index_patterns": ["station-alerts*"],
        "template": {
            "settings": {
                "number_of_shards": 2,
                "number_of_replicas": 0
            }
        }
    }
    es.indices.put_index_template(name="station_alerts_template", body=template_body)
    if not es.indices.exists(index="station-alerts"):
        es.indices.create(index="station-alerts", body={"settings": {"number_of_shards": 2, "number_of_replicas": 0}})
        print("[+] Created 'station-alerts' index with 2 primary shards.")
    else:
        print("[+] 'station-alerts' index ready (2 shards via template).")
except Exception as e:
    print(f"[-] ES pre-flight notice: {e}")

### Step 1: Load Trained MLlib Pipeline Model

In [ ]:
if not os.path.exists(model_path):
    raise FileNotFoundError(f"Model not found at {model_path}. Run notebook 02 first!")

model = PipelineModel.load(model_path)
print("MLlib Anomaly PipelineModel successfully loaded into memory!")

### Step 2: Define Schema & Subscribe to Kafka Stream

In [ ]:
telemetry_schema = StructType([
    StructField("timestamp", StringType(), True),
    StructField("station_id", StringType(), True),
    StructField("location", StringType(), True),
    StructField("voltage_v", DoubleType(), True),
    StructField("current_a", DoubleType(), True),
    StructField("temperature_c", DoubleType(), True),
    StructField("power_kw", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("error_code", StringType(), True)
])

df_raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", kafka_server) \
    .option("subscribe", input_topic) \
    .option("startingOffsets", "latest") \
    .load()

df_parsed = df_raw.selectExpr("CAST(value AS STRING) as json_str") \
    .select(from_json(col("json_str"), telemetry_schema).alias("data")) \
    .select("data.*")

print("Streaming DataFrame schema:")
df_parsed.printSchema()

### Step 3: Apply ML Inference & Filter Anomalies
VectorAssembler and RandomForest scoring happen in RAM for each incoming micro-batch.

In [ ]:
df_scored = model.transform(df_parsed)

# Filter critical anomalies
df_anomalies = df_scored.filter(
    (col("prediction") == 1.0) | (col("temperature_c") > 75.0) | (col("status") == "FAULT")
)

### Step 4: Start Streaming Sinks (Kafka & Elasticsearch)
Writes detected anomalies back to Kafka topic `station-alerts` and directly to Elasticsearch.

In [ ]:
def process_batch(batch_df, batch_id):
    count = batch_df.count()
    if count == 0:
        return
    print(f"\n[!] >>> BATCH {batch_id}: Detected {count} CRITICAL ANOMALIES! <<<")
    batch_df.select("timestamp", "station_id", "location", "temperature_c", "voltage_v", "prediction").show(5, truncate=False)
    
    try:
        from elasticsearch import Elasticsearch, helpers
        es = Elasticsearch("http://elasticsearch:9200")
        records = batch_df.toPandas().to_dict(orient="records")
        actions = [
            {
                "_index": "station-alerts",
                "_source": {
                    "timestamp": r["timestamp"],
                    "station_id": r["station_id"],
                    "location": r["location"],
                    "voltage_v": float(r["voltage_v"]),
                    "current_a": float(r["current_a"]),
                    "temperature_c": float(r["temperature_c"]),
                    "power_kw": float(r["power_kw"]),
                    "status": r["status"],
                    "error_code": r["error_code"],
                    "anomaly_prediction": int(r["prediction"]),
                    "alert_level": "CRITICAL" if r["temperature_c"] > 80.0 else "WARNING"
                }
            }
            for r in records
        ]
        helpers.bulk(es, actions)
        print(f"[+] Indexed {len(actions)} alert documents into Elasticsearch index 'station-alerts'!")
    except Exception as e:
        print(f"[-] ES Index notice: {e}")

# Kafka Alert Sink
kafka_alerts = df_anomalies.select(
    col("station_id").alias("key"),
    to_json(struct(col("*"))).alias("value")
).writeStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", kafka_server) \
    .option("topic", output_topic) \
    .option("checkpointLocation", "/tmp/checkpoints/kafka_alerts") \
    .outputMode("append") \
    .trigger(processingTime="5 seconds") \
    .start()

# Console & ES Sink
query_es = df_anomalies.writeStream \
    .foreachBatch(process_batch) \
    .outputMode("append") \
    .trigger(processingTime="5 seconds") \
    .start()

print("Streaming Queries actively running! Check Kibana at http://localhost:5601")